In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
from ast import literal_eval
from collections import Counter
import math
import numpy as np
import seaborn as sns

In [2]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [3]:
benchmarks = [
    # Low-sharing Apps
    "fdtd2d",
    "jacobi1d",
    "fastwalshtransform",
    "relu",
    "fft",
    "fir",
    "sssp",
    "convolution2d",
    "color",
    "bitonicsort",
    "pagerank",
    "simpleconvolution",

    # High-sharing Apps
    'gups',    
    'syr2k',
    'spmv',
    'mis', 
    'stencil2d',
    'gemm',
    'lu',
    'matrixmultiplication',
    'mm2',
    'mm3',
    'gaussian',
    'matrixtranspose',
    'im2col',
    'vgg16',
    'bfs',
]

In [9]:
num_chiplets = 36
ns = 0.000000001
chip_cols = [str(i) for i in range(36)]

In [10]:
base_dir=f'{username}/mgvm/results/memory-traces'
date = "20260221-223953"
expanded_dir = f"{base_dir}/{date}/expanded"
trace_dir = f"{base_dir}/{date}"
print(f"{base_dir}/{date}")

/home/ehdgus7279/mgvm/results/memory-traces/20260221-223953


In [11]:
benchmark = 'syr2k'
trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"

df = pd.read_csv(trace_file)
df

,time,pc,addr,owner,rw,0,1,2,3,4,...,28,29,30,31,32,33,34,35,access_count,shared_chiplets
0,0.023207,571330816,2718494720,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
1,0.023207,571330816,2718494720,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
2,0.023207,571330816,2718494720,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
3,0.023207,571330816,2718494720,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
4,0.023207,571330816,2718494720,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28167130,0.039706,571331132,2413686720,29,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,1
28167131,0.039706,571331040,2614697984,1,0,0,0,0,0,0,...,0,2,0,0,0,0,0,0,2,1
28167132,0.039706,571331140,2352771840,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
28167133,0.039706,571331040,2626461824,0,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,1,1


In [40]:
agg_dict = {col: 'sum' for col in chip_cols}
agg_dict['access_count'] = 'sum'
agg_dict['owner'] = 'first'
agg_dict['rw'] = 'max'

grouped = df.groupby('addr').agg(agg_dict).reset_index()

In [43]:
owner_chips = grouped['owner']
owner_chips = owner_chips.astype(str)
owner_chips

0         0
1         0
2         0
3         0
4         0
         ..
307167    1
307168    1
307169    1
307170    1
307171    1
Name: owner, Length: 307172, dtype: str

In [50]:
grouped

,addr,0,1,2,3,4,5,6,7,8,...,29,30,31,32,33,34,35,access_count,owner,rw
0,2147487744,283,6,6,6,6,6,6,6,6,...,6,6,6,6,6,6,5,492,0,0
1,2147487808,113,6,6,6,6,6,6,6,6,...,6,6,6,6,6,6,8,325,0,0
2,2147487872,85,6,7,6,10,6,7,9,9,...,6,6,9,10,7,7,10,358,0,0
3,2147487936,77,12,20,17,25,16,20,15,29,...,12,29,16,23,20,17,8,749,0,0
4,2147488000,53,29,38,22,35,27,29,24,29,...,24,29,17,32,29,38,13,1056,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307167,2718539456,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,9,1,1
307168,2718539520,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,21,1,1
307169,2718539584,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,7,1,1
307170,2718539648,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,24,1,1


In [52]:
import numpy as np

# 1. 0~35 컬럼명만 안전하게 추출
valid_chip_strs = {str(i) for i in range(36)}
chip_cols = [col for col in grouped.columns if str(col) in valid_chip_strs]

# 2. owner 값을 "순수 NumPy 문자열 배열"로 변환 후 2차원(N, 1)으로 세움
owner_vals = np.array(grouped['owner'], dtype=str)[:, None]  # Shape: (307172, 1)

# 3. 컬럼명 배열도 "순수 NumPy 문자열 배열"로 변환
col_vals = np.array(chip_cols, dtype=str)  # Shape: (36,)

# 4. 브로드캐스팅 비교 (Pandas 간섭 없이 순수 NumPy C엔진으로 초고속 연산됨)
is_owner = owner_vals == col_vals  # Shape: (307172, 36) 의 True/False 마스크 생성

# 5. 마스크 적용하여 값 0으로 치환
grouped[chip_cols] = grouped[chip_cols].mask(is_owner, 0)


In [53]:
grouped

,addr,0,1,2,3,4,5,6,7,8,...,29,30,31,32,33,34,35,access_count,owner,rw
0,2147487744,0,6,6,6,6,6,6,6,6,...,6,6,6,6,6,6,5,492,0,0
1,2147487808,0,6,6,6,6,6,6,6,6,...,6,6,6,6,6,6,8,325,0,0
2,2147487872,0,6,7,6,10,6,7,9,9,...,6,6,9,10,7,7,10,358,0,0
3,2147487936,0,12,20,17,25,16,20,15,29,...,12,29,16,23,20,17,8,749,0,0
4,2147488000,0,29,38,22,35,27,29,24,29,...,24,29,17,32,29,38,13,1056,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307167,2718539456,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,9,1,1
307168,2718539520,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,21,1,1
307169,2718539584,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,7,1,1
307170,2718539648,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,24,1,1


In [ ]:
filtered_grouped = grouped[(grouped[chip_cols] != 0).any(axis=1)]
filtered_grouped

,addr,0,1,2,3,4,5,6,7,8,...,29,30,31,32,33,34,35,access_count,owner,rw
0,2147487744,0,6,6,6,6,6,6,6,6,...,6,6,6,6,6,6,5,492,0,0
1,2147487808,0,6,6,6,6,6,6,6,6,...,6,6,6,6,6,6,8,325,0,0
2,2147487872,0,6,7,6,10,6,7,9,9,...,6,6,9,10,7,7,10,358,0,0
3,2147487936,0,12,20,17,25,16,20,15,29,...,12,29,16,23,20,17,8,749,0,0
4,2147488000,0,29,38,22,35,27,29,24,29,...,24,29,17,32,29,38,13,1056,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307167,2718539456,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,9,1,1
307168,2718539520,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,21,1,1
307169,2718539584,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,7,1,1
307170,2718539648,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,24,1,1


In [65]:
grouped['addr'].nunique()

307172

In [66]:
filtered_grouped['addr'].nunique()

188206

In [67]:
filtered_grouped[filtered_grouped['rw'] == 0]['addr'].nunique()

8915

In [68]:
filtered_grouped[filtered_grouped['rw'] == 1]['addr'].nunique()

179291

In [69]:
filtered_grouped[filtered_grouped['rw'] == 0]['access_count'].sum()

np.int64(5004441)

In [70]:
filtered_grouped[filtered_grouped['rw'] == 1]['access_count'].sum()

np.int64(17094132)

In [71]:
grouped['access_count'].sum()

np.int64(29240686)

In [73]:
filtered_grouped

,addr,0,1,2,3,4,5,6,7,8,...,29,30,31,32,33,34,35,access_count,owner,rw
0,2147487744,0,6,6,6,6,6,6,6,6,...,6,6,6,6,6,6,5,492,0,0
1,2147487808,0,6,6,6,6,6,6,6,6,...,6,6,6,6,6,6,8,325,0,0
2,2147487872,0,6,7,6,10,6,7,9,9,...,6,6,9,10,7,7,10,358,0,0
3,2147487936,0,12,20,17,25,16,20,15,29,...,12,29,16,23,20,17,8,749,0,0
4,2147488000,0,29,38,22,35,27,29,24,29,...,24,29,17,32,29,38,13,1056,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307167,2718539456,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,9,1,1
307168,2718539520,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,21,1,1
307169,2718539584,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,7,1,1
307170,2718539648,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,24,1,1


In [75]:
non_shared = grouped[chip_cols].sum(axis=1)
non_shared

0          209
1          212
2          273
3          672
4         1003
          ... 
307167       9
307168      21
307169       7
307170      24
307171      11
Length: 307172, dtype: int64

In [78]:
grouped[grouped[chip_cols].sum(axis=1).values == 0]

,addr,0,1,2,3,4,5,6,7,8,...,29,30,31,32,33,34,35,access_count,owner,rw
10,2147488384,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,35,0,0
11,2147488448,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,35,0,0
12,2147488512,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,33,0,0
13,2147488576,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,25,0,0
14,2147488640,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,16,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306983,2718527680,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,13,1,1
306984,2718527744,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,44,1,1
306985,2718527808,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,12,1,1
306986,2718527872,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,37,1,1
